# CARE Patch Generation (Real Data, Nested Layout)

This notebook generates training patches (`.npz`) for CARE / csbdeep training from *real* microscopy datasets stored in a nested folder structure. **Specifically for data from ISS preprocessing pipeline**. 

### What it does
For every detected sample folder, it:
1. Finds matching pairs between a **source** directory (e.g. `raw`) and a **target** directory (e.g. `rlf50` or `rlf25`) by **relative path**.
2. Splits data into two patch datasets:
   - **NON_DAPI**: all channels except the user-specified DAPI channel index
   - **DAPI_ONLY**: only the DAPI channel
3. Excludes files that do **not** contain a `_chN.tif` pattern.
4. Runs `csbdeep.data.create_patches(...)` to create patches and saves `.npz` files.

### Expected folder layout (per sample)
A *sample folder* is any directory that contains both `SOURCE_DIRNAME` and `TARGET_DIRNAME`.

`<CARE_ROOT>/<any_structure>/<sample>/{SOURCE_DIRNAME,TARGET_DIRNAME}/R*/preprocessing/Cycle*/4_retiled/*.tif`

Example:
- `/home/sagah/moldia-archive/CARE_training_data/Victoria/BCR_TCR_CDR3/tonsil/<sample>/raw/R1/preprocessing/Cycle1/4_retiled/Cycle1_s0_ch0.tif`
- `/home/sagah/moldia-archive/CARE_training_data/Victoria/BCR_TCR_CDR3/tonsil/<sample>/rlf50/R1/preprocessing/Cycle1/4_retiled/Cycle1_s0_ch0.tif`

Matching is done by relative path, e.g.:
- `raw/R1/preprocessing/Cycle1/4_retiled/Cycle1_s0_ch0.tif`
- `rlf50/R1/preprocessing/Cycle1/4_retiled/Cycle1_s0_ch0.tif`

### Outputs
Per sample (stored in `<sample>/train_patches/`):
- `<group>__<sample>__NON_DAPI__train_patches.npz`
- `<group>__<sample>__DAPI_ONLY__train_patches.npz`

Optional merged outputs (stored in `<CARE_ROOT>/train_patches/`):
- `ALL_SAMPLES__NON_DAPI__train_patches.npz`
- `ALL_SAMPLES__DAPI_ONLY__train_patches.npz`

### Next step
Use the produced `.npz` file(s) in the CARE training notebook (e.g. train main model on NON_DAPI, and train a separate model on DAPI_ONLY).

## Imports

In [1]:
from pathlib import Path
from ISS_CARE.ISS_CARE_datagen import run_patch_generation, visualize_saved_patches_across_samples


## User settings

Edit all user-configurable parameters in the **User settings** cell at the top of the notebook.

These include:

- `CARE_ROOT`: root folder containing all CARE training data  
- `CARE_SUBDIRS`: top-level subdirectories to process  
      - set to `[]` to include **all subdirectories under `CARE_ROOT`**  
      - or specify manually, e.g. `["christina", "Victoria"]`
- `SOURCE_DIRNAME` / `TARGET_DIRNAME`: folder names inside each sample directory  
- `PATTERN`: recursive pattern used to find image tiles  
- `DAPI_CHANNEL_INDEX`: 0-based channel index for DAPI  

### Sampling settings (per sample)
- `MAX_IMAGES_PER_SAMPLE`: fixed number of image pairs to use per sample (e.g. `200–600`)  
- `FRACTION_IMAGES_PER_SAMPLE`: fraction of image pairs to use (e.g. `0.25` for 25%)
   
    - If both are set, `MAX_IMAGES_PER_SAMPLE` takes priority  
    - Set both to `None` to use all images  
    - Tip:  
        - use smaller values (e.g. `100–300`) for quick testing  
        - use larger values (e.g. `500+`) for final training  

### Patch settings

- `PATCH_SIZE`: size of extracted patches (e.g. `128x128`)  

- `N_PATCHES_PER_IMAGE`: number of patches sampled per image pair  
  - typical: `3–8` for most datasets  
  - `1–5` → good for sparse / punctate microscopy (your case)  
  - fewer = faster, less redundancy, often **better generalization**  
  - more = larger dataset, but often adds **redundant or low-information patches**  
  - ⚠️ too high → model overfits to background / easy regions  

- `PATCH_FILTER_THRESHOLD`: threshold for filtering background patches  

  Recommended values:
  - `0.0` → no filtering (keeps many background patches)  
  - `0.01` → light filtering (safe baseline)  
  - `0.02–0.05` → ✅ **recommended for your data** (better signal-to-noise in training)  
  - `≥ 0.1` → ⚠️ often too aggressive (may remove real but dim signal)


> **Note on filtering (important for your pipeline):**  
> This pipeline already performs **image-level filtering before patch extraction**, removing pairs where the data is:
> - empty or nearly empty  
> - constant / near-constant  
> - invalid (NaN / Inf)  
>
> So by the time patches are created:
> - your dataset is already **clean at the image level**  
> - `PATCH_FILTER_THRESHOLD` only controls **local patch content**
>
> In practice:
> - low values (0.01) = safer, more data  
> - moderate values (0.02–0.05) = **better training signal (recommended)**  
> - high values (>0.1) = risk of removing useful biological structure  


> **Practical recommendation for this dataset:**  
> Start with:
> ```python
> N_PATCHES_PER_IMAGE = 5
> PATCH_FILTER_THRESHOLD = 0.05
> ```
>
> This typically:
> - reduces background bias  
> - improves denoising quality  
> - avoids over-representing empty regions  
### Axes settings
- `AXES`, `PATCH_AXES`: axis conventions for input images and patches  

### Output settings
- `PATCH_DIRNAME`: folder where patch files are stored  
- `MERGE_ALL_SAMPLES`: whether to combine all samples into one dataset  
- `MERGED_NON_DAPI_NAME`, `MERGED_DAPI_NAME`: filenames for merged outputs  

In [3]:
# ----------------------------
# User settings
# ----------------------------

# Root folder containing all CARE training data
HOME = Path.home()
CARE_ROOT = HOME / "moldia-archive" / "CARE_training_data"

# Optional: restrict processing to specific top-level subdirectories under CARE_ROOT
# Set to [] to include all subdirectories under CARE_ROOT
CARE_SUBDIRS = ["christina", "Victoria", "maria_e", "yimin"]   # e.g. [] or ["christina", "Victoria"]

# Folder names inside each sample directory
SOURCE_DIRNAME = "raw"
TARGET_DIRNAME = "rlf50"       # e.g. "rlf25"

# Recursive pattern used to find tiles inside raw/ and target/
PATTERN = "**/4_retiled/*.tif"

# DAPI channel index (0-based)
DAPI_CHANNEL_INDEX = 4         # e.g. 4 for files like *_ch4.tif


# ----------------------------
# Sampling settings (per sample)
# ----------------------------

# Option A: fixed number of image pairs per sample
MAX_IMAGES_PER_SAMPLE = 500    # e.g. 200, or None to use all images

# Option B: fraction of image pairs per sample
FRACTION_IMAGES_PER_SAMPLE = None   # e.g. 0.25 for 25%

# If both are set, MAX_IMAGES_PER_SAMPLE takes priority


# ----------------------------
# Patch generation settings
# ----------------------------
PATCH_SIZE = (128, 128)
N_PATCHES_PER_IMAGE = 10
PATCH_FILTER_THRESHOLD = 0.1  # 0.0 disables filtering

# Axes settings
AXES = "YX"
PATCH_AXES = "YX"


# ----------------------------
# Output settings
# ----------------------------
PATCH_DIRNAME = "train_patches"

# Optional: merge all processed samples into one file per model variant
MERGE_ALL_SAMPLES = True
MERGED_NON_DAPI_NAME = "ALL_SAMPLES__NON_DAPI__train_patches.npz"
MERGED_DAPI_NAME     = "ALL_SAMPLES__DAPI_ONLY__train_patches.npz"


# ----------------------------
# Print settings (sanity check)
# ----------------------------
print("Home directory:", HOME)
print("CARE_ROOT:", CARE_ROOT)
print("CARE_SUBDIRS:", CARE_SUBDIRS if CARE_SUBDIRS else "[] (all subdirectories under CARE_ROOT)")
print("SOURCE_DIRNAME:", SOURCE_DIRNAME)
print("TARGET_DIRNAME:", TARGET_DIRNAME)
print("PATTERN:", PATTERN)
print("DAPI_CHANNEL_INDEX:", DAPI_CHANNEL_INDEX)

print("\nSampling settings:")
print("MAX_IMAGES_PER_SAMPLE:", MAX_IMAGES_PER_SAMPLE)
print("FRACTION_IMAGES_PER_SAMPLE:", FRACTION_IMAGES_PER_SAMPLE)

print("\nPatch settings:")
print("PATCH_SIZE:", PATCH_SIZE)
print("N_PATCHES_PER_IMAGE:", N_PATCHES_PER_IMAGE)
print("PATCH_FILTER_THRESHOLD:", PATCH_FILTER_THRESHOLD)
print("AXES:", AXES)
print("PATCH_AXES:", PATCH_AXES)

print("\nOutput settings:")
print("PATCH_DIRNAME:", PATCH_DIRNAME)
print("MERGE_ALL_SAMPLES:", MERGE_ALL_SAMPLES)
print("MERGED_NON_DAPI_NAME:", MERGED_NON_DAPI_NAME)
print("MERGED_DAPI_NAME:", MERGED_DAPI_NAME)

Home directory: /home/sagah
CARE_ROOT: /home/sagah/moldia-archive/CARE_training_data
CARE_SUBDIRS: ['christina', 'Victoria', 'maria_e', 'yimin']
SOURCE_DIRNAME: raw
TARGET_DIRNAME: rlf50
PATTERN: **/4_retiled/*.tif
DAPI_CHANNEL_INDEX: 4

Sampling settings:
MAX_IMAGES_PER_SAMPLE: 500
FRACTION_IMAGES_PER_SAMPLE: None

Patch settings:
PATCH_SIZE: (128, 128)
N_PATCHES_PER_IMAGE: 10
PATCH_FILTER_THRESHOLD: 0.1
AXES: YX
PATCH_AXES: YX

Output settings:
PATCH_DIRNAME: train_patches
MERGE_ALL_SAMPLES: True
MERGED_NON_DAPI_NAME: ALL_SAMPLES__NON_DAPI__train_patches.npz
MERGED_DAPI_NAME: ALL_SAMPLES__DAPI_ONLY__train_patches.npz


## Run patch generation

Run this cell to generate CARE training patches from your data.

This step will:

- automatically detect all valid **sample directories**
- match `raw` and `target` images by **relative path**
- optionally **subsample images per sample**
- split data into:
  - **NON_DAPI** (main model)
  - **DAPI_ONLY** (separate model)
- generate training patches using `csbdeep`
- save `.npz` patch files per sample
- optionally create **merged datasets across all samples**
- automatically generate and save a **metadata file** with all parameters and outputs

Outputs include:
- per-sample patch files in `<sample>/train_patches/`
- optional merged datasets in `<CARE_ROOT>/train_patches/`
- metadata file:  
  `<CARE_ROOT>/train_patches/patch_generation_metadata.json`

⚠️ Depending on dataset size, this step can take time.

In [ ]:
results = run_patch_generation(
    care_root=CARE_ROOT,
    care_subdirs=CARE_SUBDIRS,
    source_dirname=SOURCE_DIRNAME,
    target_dirname=TARGET_DIRNAME,
    pattern=PATTERN,
    dapi_channel_index=DAPI_CHANNEL_INDEX,
    axes=AXES,
    patch_size=PATCH_SIZE,
    n_patches_per_image=N_PATCHES_PER_IMAGE,
    patch_axes=PATCH_AXES,
    patch_filter_threshold=PATCH_FILTER_THRESHOLD,
    patch_dirname=PATCH_DIRNAME,
    merge_all_samples=MERGE_ALL_SAMPLES,
    merged_non_dapi_name=MERGED_NON_DAPI_NAME,
    merged_dapi_name=MERGED_DAPI_NAME,
    max_images_per_sample=MAX_IMAGES_PER_SAMPLE,
    fraction_images_per_sample=FRACTION_IMAGES_PER_SAMPLE,
)



Starting CARE patch generation
Dataset root: /home/sagah/moldia-archive/CARE_training_data
Search subdirectories: ['christina', 'Victoria', 'maria_e', 'yimin']
Source folder name: raw
Target folder name: rlf50
File pattern: **/4_retiled/*.tif
DAPI channel index (0-based): 4
Max images per sample: 500
Fraction of images per sample: None
Patch filter threshold: 0.1
Minimum source max intensity: 0.0
Minimum source std: 1e-06
Minimum target max intensity: 0.0
Minimum target std: 1e-06
Extreme value cutoff: 1000000.0
Sample directories detected: 7

[Sample 1/7] Processing: 20250116_tonsil_S14_VDJ_new
Samples remaining after this one: 6
Sample directory: /home/sagah/moldia-archive/CARE_training_data/Victoria/BCR_TCR_CDR3/tonsil/20250116_tonsil_S14_VDJ_new
Sample group: Victoria__BCR_TCR_CDR3__tonsil
Stage 1/4: Matching source and target files
Stage 2/4: Sampling and image-level filtering
Pair summary:
  Total matched source/target pairs: 1470
  Available before sampling:
    NON_DAPI: 1225


100%|████████████████████████████████████████████████████████████████████████████████████| 403/403 [25:17<00:00,  3.76s/it]


Saving data to /home/sagah/moldia-archive/CARE_training_data/Victoria/BCR_TCR_CDR3/tonsil/20250116_tonsil_S14_VDJ_new/train_patches/Victoria__BCR_TCR_CDR3__tonsil__20250116_tonsil_S14_VDJ_new__NON_DAPI__train_patches.npz.
Post-create_patches summary for: Victoria__BCR_TCR_CDR3__tonsil__20250116_tonsil_S14_VDJ_new__NON_DAPI__train_patches.npz
  X -> dtype: float32, shape: (4030, 1, 128, 128), min: -1.41026, max: 28.8907, absmax: 28.8907
  Y -> dtype: float32, shape: (4030, 1, 128, 128), min: -0.586538, max: 255.273, absmax: 255.273
Finished NON_DAPI patch creation.
  X shape: (4030, 1, 128, 128)
  Y shape: (4030, 1, 128, 128)
  Axes: SCYX
Creating DAPI_ONLY patches...
Output file: /home/sagah/moldia-archive/CARE_training_data/Victoria/BCR_TCR_CDR3/tonsil/20250116_tonsil_S14_VDJ_new/train_patches/Victoria__BCR_TCR_CDR3__tonsil__20250116_tonsil_S14_VDJ_new__DAPI_ONLY__train_patches.npz
  197 raw images x    1 transformations   =   197 images
  197 images     x   10 patches per image =  19

100%|████████████████████████████████████████████████████████████████████████████████████| 197/197 [12:00<00:00,  3.66s/it]


Saving data to /home/sagah/moldia-archive/CARE_training_data/Victoria/BCR_TCR_CDR3/tonsil/20250116_tonsil_S14_VDJ_new/train_patches/Victoria__BCR_TCR_CDR3__tonsil__20250116_tonsil_S14_VDJ_new__DAPI_ONLY__train_patches.npz.
Post-create_patches summary for: Victoria__BCR_TCR_CDR3__tonsil__20250116_tonsil_S14_VDJ_new__DAPI_ONLY__train_patches.npz
  X -> dtype: float32, shape: (1970, 1, 128, 128), min: -0.316038, max: 13.4512, absmax: 13.4512
  Y -> dtype: float32, shape: (1970, 1, 128, 128), min: -0.408759, max: 16.6144, absmax: 16.6144
Finished DAPI_ONLY patch creation.
  X shape: (1970, 1, 128, 128)
  Y shape: (1970, 1, 128, 128)
  Axes: SCYX

[Sample 2/7] Processing: CHK_pool1-2_SplintR_B2DO27_40X_Slide4
Samples remaining after this one: 5
Sample directory: /home/sagah/moldia-archive/CARE_training_data/christina/Chicken/CHK_pool1-2_SplintR_B2DO27_40X_Slide4
Sample group: christina__Chicken
Stage 1/4: Matching source and target files
Stage 2/4: Sampling and image-level filtering
Pair su

...

Saving data to /home/sagah/moldia-archive/CARE_training_data/maria_e/Cutcancer-biopsies-pat1-sample2_2/train_patches/maria_e__Cutcancer-biopsies-pat1-sample2_2__DAPI_ONLY__train_patches.npz.
Post-create_patches summary for: maria_e__Cutcancer-biopsies-pat1-sample2_2__DAPI_ONLY__train_patches.npz
  X -> dtype: float32, shape: (960, 1, 128, 128), min: -0.00930233, max: 6.96786, absmax: 6.96786
  Y -> dtype: float32, shape: (960, 1, 128, 128), min: -0.0226537, max: 26.4302, absmax: 26.4302
Finished DAPI_ONLY patch creation.
  X shape: (960, 1, 128, 128)
  Y shape: (960, 1, 128, 128)
  Axes: SCYX

[Sample 7/7] Processing: SCRINSHOT_4cycles
Samples remaining after this one: 0
Sample directory: /home/sagah/moldia-archive/CARE_training_data/yimin/SCRINSHOT_4cycles
Sample group: yimin
Stage 1/4: Matching source and target files
Stage 2/4: Sampling and image-level filtering
Pair summary:
  Total matched source/target pairs: 960
  Available before sampling:
    NON_DAPI: 800
    DAPI_ONLY: 160
 

100%|████████████████████████████████████████████████████████████████████████████████████| 473/473 [25:52<00:00,  3.28s/it]


Saving data to /home/sagah/moldia-archive/CARE_training_data/yimin/SCRINSHOT_4cycles/train_patches/yimin__SCRINSHOT_4cycles__NON_DAPI__train_patches.npz.
Post-create_patches summary for: yimin__SCRINSHOT_4cycles__NON_DAPI__train_patches.npz
  X -> dtype: float32, shape: (4730, 1, 128, 128), min: -0.148649, max: 112.707, absmax: 112.707
  Y -> dtype: float32, shape: (4730, 1, 128, 128), min: -0.113894, max: 427.51, absmax: 427.51
Finished NON_DAPI patch creation.
  X shape: (4730, 1, 128, 128)
  Y shape: (4730, 1, 128, 128)
  Axes: SCYX
Creating DAPI_ONLY patches...
Output file: /home/sagah/moldia-archive/CARE_training_data/yimin/SCRINSHOT_4cycles/train_patches/yimin__SCRINSHOT_4cycles__DAPI_ONLY__train_patches.npz
  152 raw images x    1 transformations   =   152 images
  152 images     x   10 patches per image =  1520 patches in total
Input data:
paired_generator
Transformations:
1 x Permute axes to YX
Patch size:
128 x 128


100%|████████████████████████████████████████████████████████████████████████████████████| 152/152 [08:00<00:00,  3.16s/it]


Saving data to /home/sagah/moldia-archive/CARE_training_data/yimin/SCRINSHOT_4cycles/train_patches/yimin__SCRINSHOT_4cycles__DAPI_ONLY__train_patches.npz.
Post-create_patches summary for: yimin__SCRINSHOT_4cycles__DAPI_ONLY__train_patches.npz
  X -> dtype: float32, shape: (1520, 1, 128, 128), min: -0.116279, max: 10.8162, absmax: 10.8162
  Y -> dtype: float32, shape: (1520, 1, 128, 128), min: -0.0359281, max: 76.2156, absmax: 76.2156
Finished DAPI_ONLY patch creation.
  X shape: (1520, 1, 128, 128)
  Y shape: (1520, 1, 128, 128)
  Axes: SCYX


## Visualize generated patches

Run this cell to inspect a few example patch pairs from each sample.

This will:

- load the saved `.npz` patch files
- randomly select a few patch pairs per sample
- display **input (top)** and **target (bottom)** images
- apply normalization for better visibility

This is useful to:
- verify that source and target are correctly aligned  
- check patch quality and signal content  
- spot potential issues before training  


In [ ]:
# Visualize all samples
visualize_saved_patches_across_samples(
                    results['all_patch_files_non_dapi'], 
                    n_show_per_sample=5)
